# HarvestWise â€” Phase 2: Model Training (Colab)

Trains the multimodal yield-forecast model (vision + weather + soil encoders â†’ phenology-aware fusion â†’ spatio-temporal Transformer â†’ quantile head).

**Honest framing of this run, read before trusting any number it prints:**
- Training data is **synthetic** (procedurally generated season curves) â€” this is a pretraining/feasibility run, not a claim of real-world accuracy.
- The **3 real Coimbatore rice fields** (Sulur, Kinathukadavu, Annur) are held out and never trained on. They're used only as a plausibility check: does the model produce sane yield numbers (roughly 2â€“5 t/ha) when given real satellite/weather/soil inputs?
- All 3 real fields currently share **one repeated yield label** (3.641 t/ha, from Tamil Nadu government 2019-20 district statistics â€” see `data/raw/yield_labels/README.md` for the documented year-gap limitation). This means the real-holdout MAE has **zero real target variance to explain** â€” it is not a statistically meaningful accuracy benchmark, and should not be reported as one in a paper without this caveat.
- A naive baseline ("always predict the training-set mean yield") is printed alongside the model's result for comparison â€” if the model doesn't beat it, that's a real, honestly-reported outcome, not a bug to hide.

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Upload the project package

Upload `HarvestWise_colab_package.zip` (built locally from the real codebase + real processed Coimbatore data) using the file picker below.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select HarvestWise_colab_package.zip

In [ ]:
import zipfile
with zipfile.ZipFile('HarvestWise_colab_package.zip') as zf:
    zf.extractall('.')
%cd HarvestWise
!ls

## 3. Install dependencies

Only what's needed for training (not the full ingestion stack â€” no earthengine-api/cdsapi/rasterio/geopandas here, Colab already ships numpy/pandas/torch).

In [ ]:
!pip install -q torch numpy pandas scikit-learn scipy

## 4. Sanity check: real held-out data loads correctly

Confirms the 3 real Coimbatore fields, their vegetation indices, weather, and (imputed where missing) soil data build into season examples before spending any GPU time.

In [ ]:
from training.dataset import build_dataset_from_processed
real_examples = build_dataset_from_processed()
print(f"real season examples: {len(real_examples)}")
for ex in real_examples:
    print(ex.field_id, ex.season_start_date, ex.final_yield, 'soil ok' if not __import__('numpy').isnan(ex.soil_x).any() else 'soil MISSING')

## 5. Train (synthetic pretrain + real held-out plausibility check)

`--mode realistic` trains only on synthetic data and evaluates against the real fields every 5 epochs without ever training on them. Bump `--epochs` and `--synthetic-n` up for a more serious run once the short version below completes cleanly.

In [ ]:
!python -m training.train_forecast_model --mode realistic --epochs 100 --synthetic-n 300 --batch-size 16

## 6. Compare against classical ML baselinesMirrors the baseline-comparison-table approach used in published ML papers (train Random Forest + XGBoost on the same data, evaluate everyone against the same real held-out set) so the deep model's result isn't reported in isolation.

In [ ]:
!pip install -q xgboost

In [ ]:
!python -m evaluation.run_model_comparison

## 7. Download the trained checkpoints

`fusion_backbone.pt` + `yield_head.pt` â€” drop these into `backend/checkpoints/` locally and the FastAPI backend switches from placeholder inference to the real trained model automatically (see `backend/app/models_registry/model_loader.py`).

In [ ]:
from google.colab import files
files.download('backend/checkpoints/fusion_backbone.pt')
files.download('backend/checkpoints/yield_head.pt')

## 8. Optional: try `--mode real` to see why it's not viable yet

This trains directly on the 12 real season examples (3 fields Ã— 4 years) with no synthetic data at all. Expect it to fail informatively or overfit immediately â€” 12 examples, one repeated label, is not enough to train a deep model from scratch. Running it once is useful evidence for a limitations section, not a real training strategy.

In [ ]:
!python -m training.train_forecast_model --mode real --epochs 30 --batch-size 4